# LoKU fair protocol (Kaggle)

Mot lan train voi ngan sach co dinh 30 epoch se tao hai ket qua: `last` va `val_best`. Checkpoint `val_best` chi duoc chon bang validation CE; test, forget metrics, MIA va gold retrained model khong tham gia chon epoch.

Mac dinh notebook chay MIMIC-CXR 3%, seed 42. De chay 6% hoac 10%, chi doi `FORGET_PCT` trong Cell 2 va chay lai Cell 2-4.

In [ ]:
# Cell 1: clean Kaggle setup
import os, subprocess

WORK_DIR = '/kaggle/working'
REPO_DIR = f'{WORK_DIR}/Forget-MI-LoKU'
REPO_URL = 'https://github.com/nhnhu146/Forget-MI-LoKU.git'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
protocol_source = open('training/forgetmi_loku.py', encoding='utf-8').read()
assert 'evaluate_last_and_best' in protocol_source, 'Repo tren GitHub chua co fair-protocol code moi.'

subprocess.run(['pip', 'install', '-q', 'pydicom', 'scikit-image', 'scikit-learn', 'pyyaml', 'wandb', 'seaborn==0.13.2'], check=True)
subprocess.run([
    'pip', 'install', '-q', 'transformers==4.38.0',
    'peft==0.10.0', 'accelerate==0.27.0'
], check=True)

import torch
assert torch.cuda.is_available(), 'Bat GPU trong Kaggle Settings truoc khi chay.'
print('Repo:', REPO_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('GPU :', torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: experiment controls and path discovery
import glob, os

FORGET_PCT = 3       # allowed: 3, 6, 10
SEED = 42
EPOCHS = 30
assert FORGET_PCT in (3, 6, 10)

def find_dataset(*slugs):
    for slug in slugs:
        direct = f'/kaggle/input/{slug}'
        if os.path.isdir(direct):
            return direct
        hits = glob.glob(f'/kaggle/input/datasets/*/{slug}')
        if hits:
            return sorted(hits)[0]
    return None

def first_existing(root, relatives):
    for rel in relatives:
        path = os.path.join(root, rel)
        if os.path.exists(path):
            return path
    return None

DATA_ROOT = find_dataset('forget-mi-data')
MODELS_ROOT = find_dataset('forget-mi-models-full', 'forget-mi-models-v2', 'forget-mi-models')
assert DATA_ROOT and MODELS_ROOT, 'Add forget-mi-data and forget-mi-models(-full) to Kaggle.'

base_hits = glob.glob(os.path.join(MODELS_ROOT, '**', 'training_original_model', 'pytorch_model.bin'), recursive=True)
gold_hits = glob.glob(os.path.join(MODELS_ROOT, '**', f'model_retrained_{FORGET_PCT}per', '**', 'pytorch_model.bin'), recursive=True)
BASE_MODEL = os.path.dirname(sorted(base_hits, key=len)[0]) if base_hits else None
GOLD_MODEL = os.path.dirname(sorted(gold_hits, key=len)[0]) if gold_hits else BASE_MODEL
TEXT_DIR = first_existing(DATA_ROOT, ['data/metadata', 'metadata'])
IMG_DIR = first_existing(DATA_ROOT, ['data/img_data', 'img_data'])
FORGET_CSV = f'./data_splits/forget_set_{FORGET_PCT}per.csv'
RESULTS_CSV = '/kaggle/working/fair_loku_results.csv'
HISTORY_CSV = f'/kaggle/working/fair_loku_history_{FORGET_PCT}per_s{SEED}.csv'
FIG_DIR = f'/kaggle/working/thesis_figures/loku_{FORGET_PCT}per_s{SEED}'
OUTPUT_DIR = f'/kaggle/working/fair_loku_output/{FORGET_PCT}per_seed{SEED}'
RUN_ID = f'fair_loku_{FORGET_PCT}per_s{SEED}'

for name, path in {
    'base model': BASE_MODEL, 'text metadata': TEXT_DIR,
    'images': IMG_DIR, 'forget csv': FORGET_CSV
}.items():
    assert path and os.path.exists(path), f'Missing {name}: {path}'

print('Forget percentage:', FORGET_PCT)
print('Base model       :', BASE_MODEL)
print('Gold retrained   :', GOLD_MODEL if gold_hits else 'N/A (CosSim will be invalid)')
print('Results CSV      :', RESULTS_CSV)


In [ ]:
# Cell 3: one LoKU run -> last + val_best
import os, subprocess

overrides = {
    'id': RUN_ID,
    'forget_set_path': FORGET_CSV,
    'base_model_path': BASE_MODEL,
    'bert_pretrained_dir': BASE_MODEL,
    'retrained_model_path': GOLD_MODEL,
    'text_data_dir': TEXT_DIR,
    'img_data_dir': IMG_DIR,
    'output_dir': OUTPUT_DIR,
    'results_csv_path': RESULTS_CSV,
    'history_csv_path': HISTORY_CSV,
    'unlearn_epochs': EPOCHS,
    'evaluate_last_and_best': 1,
    'selection_max_validation': 400,
    'selection_min_delta': 0.0,
    'early_stop_metric': 'val',
    'early_stopping_enabled': 0,
}
if os.path.exists(HISTORY_CSV):
    os.remove(HISTORY_CSV)
override_arg = ','.join(f'{key}={value}' for key, value in overrides.items())
env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_MODE': 'disabled'}
cmd = [
    'python', 'training/forgetmi_loku.py',
    '--config', 'config_loku_kaggle.yaml',
    '--seed', str(SEED), '--fresh', '--override', override_arg
]
print('Running:', RUN_ID)
subprocess.run(cmd, env=env, check=True)


In [ ]:
# Cell 4: verify the two result rows
import pandas as pd

df = pd.read_csv(RESULTS_CSV)
rows = df[(df['id'] == RUN_ID) & (df['seed'] == SEED)].tail(2).copy()
assert set(rows['checkpoint']) == {'last', 'val_best'}, rows
columns = [
    'method', 'checkpoint', 'selected_epoch', 'selection_value',
    'MIA_paper', 'Df_AUC', 'Df_F1', 'Dt_AUC', 'Dt_F1',
    'fisher_hours', 'adapter_init_hours', 'train_hours',
    'selection_hours', 'unlearn_core_hours', 'unlearn_total_hours',
    'trainable_params', 'trainable_ratio', 'gpu_peak_GB', 'gpu_name'
]
display(rows[[c for c in columns if c in rows.columns]])
print('OK: one training run produced last and val_best.')


In [ ]:
# Cell 5: thesis figures (display + 300-DPI PNG + vector PDF)
from scripts.plot_fair_thesis import generate_thesis_figures

rows, history, cross_method = generate_thesis_figures(
    results_csv=RESULTS_CSV,
    history_csv=HISTORY_CSV,
    run_id=RUN_ID,
    seed=SEED,
    fig_dir=FIG_DIR,
    comparison_csvs=[
        '/kaggle/working/fair_forgetmi_results.csv',
        '/kaggle/working/fair_loku_results.csv',
    ],
)
import shutil
FIG_ARCHIVE = shutil.make_archive(FIG_DIR, 'zip', root_dir=FIG_DIR)
print('Figure archive:', FIG_ARCHIVE)
